# Day 6: Vector Databases
Your embeddings live in memory.

Restart your script and they're gone.

Vector databases persist embeddings and make search fast at scale.

In [2]:
pip install chromadb

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 21.6 MB 689 kB/s eta 0:00:01
     |████████████████████████████████| 69 kB 1.4 MB/s eta 0:00:01
     |████████████████████████████████| 40 kB 1.9 MB/s eta 0:00:01
     |████████████████████████████████| 48 kB 4.4 MB/s eta 0:00:01
     |████████████████████████████████| 180 kB 1.2 MB/s eta 0:00:01
     |████████████████████████████████| 90 kB 478 kB/s eta 0:00:01
     |████████████████████████████████| 174 kB 525 kB/s eta 0:00:01
     |████████████████████████████████| 56 kB 289 kB/s eta 0:00:01
     |████████████████████████████████| 245 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 60 kB 1.6 MB/s eta 0:00:01
     |████████████████████████████████| 16.8 MB 1.2 MB/s eta 0:00:01
     |████████████████████████████████| 68 kB 3.0 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 3.4 MB/s eta 0:00:01
     |████████████████████████████████| 12.0

## Setup

In [3]:
# pip install chromadb
import chromadb
from google import genai
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)


/Users/atharva/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/atharva/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/atharva/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and t

## Create a Vector Database

In [4]:
#  Create a persistent database (saves to disk)
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Create a collection (like a table)
collection = chroma_client.get_or_create_collection(name="my_documents")

print(f"✅ Collection created: {collection.name}")

✅ Collection created: my_documents


## Add Documents

In [5]:
documents = [
    "Python is a popular programming language for data science.",
    "JavaScript powers most websites and web applications.",
    "Machine learning models learn patterns from data.",
    "Docker containers package applications for deployment.",
    "Neural networks are inspired by the human brain."
]

# Generate embeddings with Gemini
embeddings = []
for doc in documents:
    response = client.models.embed_content(model="gemini-embedding-001", contents=doc)
    embeddings.append(response.embeddings[0].values)

# Add to ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print(f"✅ Added {len(documents)} documents to the database")

✅ Added 5 documents to the database


## Search the Database

In [6]:
query = "How do AI systems learn?"

# Embed the query
query_embedding = client.models.embed_content(
    model="gemini-embedding-001", 
    contents=query
).embeddings[0].values

# Search (ChromaDB handles the similarity calculation)
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

print(f"🔎 Query: '{query}'\n")
print("📄 Results:")
for doc in results['documents'][0]:
    print(f"  • {doc}")

🔎 Query: 'How do AI systems learn?'

📄 Results:
  • Machine learning models learn patterns from data.
  • Neural networks are inspired by the human brain.


## The Key Difference

**Without a vector DB:**

embeddings = []  # Lives in memory

Script restarts → embeddings gone

1M documents → out of memory

**With a vector DB:**

collection.add(...)  # Saved to disk

Script restarts → data persists

1M documents → optimized indexing



## Key Takeaways
Vector DBs persist embeddings to disk

They handle similarity search for you

They scale to millions of documents

Popular options: ChromaDB, Pinecone, Weaviate, FAISS
